In [3]:
import time
import pandas as pd
import numpy as np

# 導入 Kernel 模組
from kernel.market_data.data_loader import MarketDataLoader
from kernel.market_data.market import Market
from kernel.market_data.volatility_surface.enums_volatility import VolatilitySurfaceType
from kernel.market_data.rate_curve_data.enums_interpolators import InterpolationType
from kernel.tools import CalendarConvention, ObservationFrequency, RateCurveType, Model
from utils.pricing_settings import PricingSettings
from kernel.pricing_launcher import PricingLauncher
from kernel.models.pricing_engines.enum_pricing_engine import PricingEngineType
from kernel.products.structured_products.autocall_products import Phoenix

pd.options.display.float_format = '{:,.4f}'.format

In [4]:
print("Initializing Market Data...")
t0 = time.time()

# 基礎 Pricing Settings
settings = PricingSettings()
settings.underlying_name = "SPX"
settings.rate_curve_type = RateCurveType.RF_US_TREASURY
settings.interpolation_type = InterpolationType.CUBIC
settings.day_count_convention = CalendarConvention.ACT_360
settings.obs_frequency = ObservationFrequency.ANNUAL
settings.volatility_surface_type = VolatilitySurfaceType.SVI
settings.compute_greeks = True
settings.nb_steps = 256

# 載入 Market Data
data_loader = MarketDataLoader()
underlying_df = data_loader.get_underlying_info(settings.underlying_name)
options_df = data_loader.get_option_data(settings.underlying_name)
yield_df = data_loader.get_yield_curve(settings.rate_curve_type.value)

market = Market(
    underlying_name=settings.underlying_name,
    yield_curve_data=yield_df,
    underlying_data=underlying_df,
    option_data=options_df,
    rate_curve_type=settings.rate_curve_type,
    interpolation_type=settings.interpolation_type,
    volatility_surface_type=settings.volatility_surface_type,
    calendar_convention=settings.day_count_convention,
    obs_frequency=settings.obs_frequency,
)

spot = market.underlying_asset.last_price
print(f"Market init time: {time.time() - t0:.4f} seconds")
print(f"Current SPX Spot Price: {spot:,.2f}")

Initializing Market Data...
Market init time: 0.0840 seconds
Current SPX Spot Price: 5,768.00


In [5]:
# 建立 Phoenix 產品
phoenix = Phoenix(
    maturity=3.0,
    observation_frequency=ObservationFrequency.SEMIANNUAL,
    capital_barrier=70.0,
    autocall_barrier=100.0,
    coupon_barrier=80.0,
    coupon_rate=8.0,
)

launcher = PricingLauncher(pricing_settings=settings, market=market)

# 定義要比較的模型情境
model_configs = [
    {"name": "BS (Pseudo)", "engine": PricingEngineType.CALLABLE_MC, "model": Model.BLACK_SCHOLES, "gen": "NUMPY", "paths": 10000},
    {"name": "BS (Sobol)", "engine": PricingEngineType.CALLABLE_MC, "model": Model.BLACK_SCHOLES, "gen": "SOBOL", "paths": 5000},
    {"name": "Heston", "engine": PricingEngineType.CALLABLE_MC, "model": Model.HESTON, "gen": "NUMPY", "paths": 10000},
]

results = []
base_greeks = {} # 留作後續 PnL 與壓力測試使用

for config in model_configs:
    settings.pricing_engine_type = config["engine"]
    settings.model = config["model"]
    settings.random_generator_type = config["gen"]
    settings.nb_paths = config["paths"]
    
    start = time.time()
    res = launcher.calculate(phoenix)
    elapsed = time.time() - start
    
    # 儲存 Heston 的結果作為基準 (Base Case)
    if config["name"] == "Heston":
        base_price = res.price
        base_greeks = res.greeks
        
    results.append({
        "Model": config["name"],
        "Price": res.price,
        "Time(s)": elapsed,
        "StdDev": res.std_dev if (res.std_dev is not None and config["gen"] != "SOBOL") else np.nan,
        "Delta": res.greeks.get('delta', np.nan) if res.greeks else np.nan,
        "Gamma": res.greeks.get('gamma', np.nan) if res.greeks else np.nan,
        "Vega": res.greeks.get('vega', np.nan) if res.greeks else np.nan,
    })

df_models = pd.DataFrame(results)
display(df_models)

,Model,Price,Time(s),StdDev,Delta,Gamma,Vega
0,BS (Pseudo),109.4452,0.3591,0.1551,0.0002,0.0000,-76.7916
1,BS (Sobol),109.3787,0.5980,NaN,0.0006,-0.0000,-85.3021
2,Heston,103.2621,0.7490,0.2089,0.0054,0.0000,-3.6870


In [6]:
# 假設 T+1 的市場變動 (Shock)
spot_shift_pct = 0.015  # SPX 上漲 1.5%
vol_shift_pts = 0.02    # Vol 上升 2% (200 bps)

delta_S = spot * spot_shift_pct
delta_Vol = vol_shift_pts

# 1. 計算 Explained PnL (基於 Greeks)
# dP ≈ Delta * dS + 0.5 * Gamma * dS^2 + Vega * dVol
pnl_delta = base_greeks.get('delta', 0) * delta_S
pnl_gamma = 0.5 * base_greeks.get('gamma', 0) * (delta_S ** 2)
pnl_vega = base_greeks.get('vega', 0) * delta_Vol

explained_pnl = pnl_delta + pnl_gamma + pnl_vega

# 2. 計算 Actual PnL (透過引擎重新定價)
# 注意：這裡假設你的 market object 與 launcher 支援直接更新 spot/vol，
# 若需重新實例化 Market，請根據你的 kernel 架構調整。
original_spot = market.underlying_asset.last_price

try:
    # 建立一個 T+1 的 Shocked Market
    market.underlying_asset.last_price = original_spot * (1 + spot_shift_pct)
    # 假設有個方法可以平移 Vol Surface (示意)
    # market.volatility_surface.shift(vol_shift_pts) 
    
    # 使用 Heston 重新計算
    settings.model = Model.HESTON
    settings.random_generator_type = "NUMPY"
    settings.nb_paths = 10000
    
    res_t1 = launcher.calculate(phoenix)
    actual_pnl = res_t1.price - base_price
    unexplained_pnl = actual_pnl - explained_pnl

    pnl_attribution = pd.DataFrame([{
        "Delta PnL": pnl_delta,
        "Gamma PnL": pnl_gamma,
        "Vega PnL": pnl_vega,
        "Total Explained": explained_pnl,
        "Actual PnL": actual_pnl,
        "Unexplained PnL": unexplained_pnl
    }])

    print("PnL Attribution (T vs T+1):")
    display(pnl_attribution)
    
finally:
    # 復原市場數據
    market.underlying_asset.last_price = original_spot

PnL Attribution (T vs T+1):


,Delta PnL,Gamma PnL,Vega PnL,Total Explained,Actual PnL,Unexplained PnL
0,0.4682,0.0553,-0.0737,0.4498,0.5047,0.0550


In [9]:
spot_shocks = [-0.10, -0.05, 0.0, 0.05, 0.10]
# 這裡僅針對 Spot 進行壓力測試
stress_results = []

settings.model = Model.HESTON
settings.nb_paths = 5000  # 壓測時可適度降低路徑數以節省時間

print("Running Stress Test Scenarios...")
for shock in spot_shocks:
    shocked_spot = original_spot * (1 + shock)
    market.underlying_asset.last_price = shocked_spot
    
    res_stress = launcher.calculate(phoenix)
    
    stress_results.append({
        "Spot Shock": f"{shock*100:2.0f}%",
        "Shocked Spot": shocked_spot,
        "Price": res_stress.price,
        "Delta": res_stress.greeks.get('delta', np.nan) if res_stress.greeks else np.nan,
        "Gamma": res_stress.greeks.get('gamma', np.nan) if res_stress.greeks else np.nan
    })

# 復原 Market
market.underlying_asset.last_price = original_spot

df_stress = pd.DataFrame(stress_results)
display(df_stress)

Running Stress Test Scenarios...


,Spot Shock,Shocked Spot,Price,Delta,Gamma
0,-10%,"5,191.2000",98.7176,0.0111,-0.0000
1,-5%,"5,479.6000",101.3851,0.0079,-0.0000
2,0%,"5,768.0000",103.3815,0.0051,0.0000
3,5%,"6,056.4000",104.6454,0.0037,-0.0000
4,10%,"6,344.8000",105.3691,0.0024,0.0000


## 6. ASC 820 Fair Value Measurement & Valuation Reserves (Product Control)
在複雜衍生性商品定價中，純理論的 Mid-Market Price 無法作為最終的 Exit Price。特別是在 Level 3 資產的查核框架下，我們必須考量避險的摩擦成本、模型不確定性，以及資金成本 (XVA)。

以下將建立一個 **Fair Value Waterfall**，模擬實務上的 Day-1 PnL 扣減：
1. **Barrier Shift Reserve**: 將 Capital Barrier 往不利方向微調 (e.g., +1%) 重新定價，差額作為對沖 Gamma 翻轉時的滑價儲備。
2. **Model Risk Reserve**: 提取 Black-Scholes 與 Heston 模型定價差異的一部分作為模型不確定性儲備。
3. **FVA (Funding Valuation Adjustment) Proxy**: 考量該部位的預期存續期間 (Expected Duration) 與交易廳無擔保資金利差 (Funding Spread)，計算資金成本。

In [10]:
print("=" * 60)
print("PRODUCT CONTROL: FAIR VALUE WATERFALL & RESERVES")
print("=" * 60)

# 確保使用 Heston 作為基礎模型
settings.model = Model.HESTON
settings.nb_paths = 10000
base_res = launcher.calculate(phoenix)
base_price = base_res.price

# ---------------------------------------------------------
# 1. Barrier Shift Reserve (模擬避險滑價與不連續性風險)
# ---------------------------------------------------------
# 將 Capital Barrier 從 70 提高到 71 (對投資人不利/對發行商有利的方向)
original_barrier = phoenix.capital_barrier
phoenix.capital_barrier = original_barrier + 1.0  

shifted_res = launcher.calculate(phoenix)
barrier_reserve = base_price - shifted_res.price

# 復原 Barrier 設定
phoenix.capital_barrier = original_barrier  

# ---------------------------------------------------------
# 2. Model Risk Reserve (模型不確定性)
# ---------------------------------------------------------
# 取得 BS (Sobol) 的價格來比較
settings.model = Model.BLACK_SCHOLES
settings.random_generator_type = "SOBOL"
bs_res = launcher.calculate(phoenix)

# 實務上可能取兩者差異的 50% 作為 Day-1 Reserve
model_reserve = abs(base_price - bs_res.price) * 0.5  

# ---------------------------------------------------------
# 3. FVA (Funding Valuation Adjustment) Proxy
# ---------------------------------------------------------
# Autocall 因為可能提前出場，有效存續期 (Effective Duration) 通常小於 Maturity
# 這裡簡單假設 Expected Duration 為 1.5 年，Funding Spread 為 50 bps
expected_duration = 1.5  
funding_spread_bps = 50 / 10000  

# FVA = PV * Funding Spread * Expected Duration (簡易估算)
fva_charge = base_price * funding_spread_bps * expected_duration

# ---------------------------------------------------------
# 4. 建立 Fair Value Waterfall (Exit Price Calculation)
# ---------------------------------------------------------
exit_price = base_price - barrier_reserve - model_reserve - fva_charge

waterfall_data = [
    {"Component": "1. Mid-Market Theoretical (Heston)", "Impact": 0.0, "Cumulative Price": base_price},
    {"Component": "2. Less: Barrier Shift Reserve (+1% Shock)", "Impact": -barrier_reserve, "Cumulative Price": base_price - barrier_reserve},
    {"Component": "3. Less: Model Risk Reserve (BS vs Heston)", "Impact": -model_reserve, "Cumulative Price": base_price - barrier_reserve - model_reserve},
    {"Component": "4. Less: FVA Charge Proxy (50 bps)", "Impact": -fva_charge, "Cumulative Price": exit_price},
    {"Component": "=> Final Exit Price (Fair Value)", "Impact": exit_price - base_price, "Cumulative Price": exit_price}
]

df_waterfall = pd.DataFrame(waterfall_data)
display(df_waterfall)

# 復原 Settings 狀態
settings.model = Model.HESTON
settings.random_generator_type = "NUMPY"

PRODUCT CONTROL: FAIR VALUE WATERFALL & RESERVES


,Component,Impact,Cumulative Price
0,1. Mid-Market Theoretical (Heston),0.0000,103.2621
1,2. Less: Barrier Shift Reserve (+1% Shock),-0.0967,103.1654
2,3. Less: Model Risk Reserve (BS vs Heston),-3.1453,100.0201
3,4. Less: FVA Charge Proxy (50 bps),-0.7745,99.2456
4,=> Final Exit Price (Fair Value),-4.0165,99.2456


In [12]:
print("=" * 60)
print("PRODUCT CONTROL: DAY-1 PNL DEFERRAL & AMORTIZATION")
print("=" * 60)

# 假設銀行以面額 (Par) 發行此 Autocall 結構型商品
# 銀行收到投資人的現金 100
issue_price = 100.0  

# 使用我們先前計算出的 Exit Price 作為公允價值負債 (Fair Value Liability)
# 假設銀行評價此負債的真實成本為 exit_price (例如 96.5)
# Day-1 經濟利潤 (Economic Gain) = 收到現金 - 承擔負債的公允價值
day1_economic_gain = issue_price - exit_price

if day1_economic_gain > 0:
    print(f"Initial Day-1 Gain: {day1_economic_gain:.4f} (Subject to Level 3 Deferral)\n")
    
    maturity_years = phoenix.maturity
    obs_freq = 2 # 半年一次觀察 (Semi-annual)
    total_periods = int(maturity_years * obs_freq)
    
    # 攤銷邏輯設定：直線攤銷 (Straight-line) 作為基礎
    amortization_per_period = day1_economic_gain / total_periods
    
    # 模擬情境：假設在第 1.5 年 (第 3 期) 發生了 Autocall 提早出場
    autocall_trigger_period = 3  
    
    schedule_data = []
    unrecognized_reserve = day1_economic_gain
    cumulative_recognized_pnl = 0.0
    
    for period in range(1, total_periods + 1):
        time_elapsed = period / obs_freq
        
        if period < autocall_trigger_period:
            # 正常存續期間，按比例攤銷
            recognized_this_period = amortization_per_period
            unrecognized_reserve -= recognized_this_period
            status = "Active (Linear Amortization)"
            
        elif period == autocall_trigger_period:
            # 發生 Autocall 提早出場，剩餘儲備「一次性全額認列」(Crystallization)
            recognized_this_period = unrecognized_reserve
            unrecognized_reserve = 0.0
            status = " Autocalled (Reserve Crystallized)"
            
        else:
            # 產品已消滅，不再有 PnL 認列
            recognized_this_period = 0.0
            status = "Terminated"
            
        cumulative_recognized_pnl += recognized_this_period
        
        schedule_data.append({
            "Period (Years)": f"{time_elapsed:.1f}",
            "Status": status,
            "Recognized PnL (This Period)": recognized_this_period,
            "Cumulative Recognized PnL": cumulative_recognized_pnl,
            "Remaining Deferred Reserve": unrecognized_reserve
        })
        
    df_amortization = pd.DataFrame(schedule_data)
    display(df_amortization)

else:
    # 基於保守穩健原則 (Conservatism principle)，Day-1 虧損必須「立即全額認列」
    print(f"Day-1 Economic Loss observed ({day1_economic_gain:.4f}).")
    print("Under accounting standards, Day-1 losses are recognized immediately in the income statement.")

PRODUCT CONTROL: DAY-1 PNL DEFERRAL & AMORTIZATION
Initial Day-1 Gain: 0.7544 (Subject to Level 3 Deferral)



,Period (Years),Status,Recognized PnL (This Period),Cumulative Recognized PnL,Remaining Deferred Reserve
0,0.5,Active (Linear Amortization),0.1257,0.1257,0.6287
1,1.0,Active (Linear Amortization),0.1257,0.2515,0.5029
2,1.5,Autocalled (Reserve Crystallized),0.5029,0.7544,0.0000
3,2.0,Terminated,0.0000,0.7544,0.0000
4,2.5,Terminated,0.0000,0.7544,0.0000
5,3.0,Terminated,0.0000,0.7544,0.0000
